In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:e4b-it-qat",
    temperature=0.2
)

print(
    llm.invoke(
        "Reply with READY"
    ).content
)

/Users/adithya/Development/adithyasean/computer-vision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


READY


In [2]:
intel_report = """
Timestamp: 2026-06-15 14:30 UTC

ISR Drone Feed:

Detected movement of 12 armored vehicles
near Northern Sector Delta.

Satellite imagery indicates
temporary encampment established
within the last 24 hours.

Communications intercepts show
encrypted radio traffic increased
by 240%.

Weather conditions favorable
for vehicle movement.

Nearby civilian settlements:
3 villages within 12km radius.
"""

In [3]:
from datetime import datetime

def run_agent(role, task):

    prompt = f"""
You are acting as:

{role}

Mission Context:

{task}

Return:
1. Findings
2. Confidence Score (0-100)
3. Concerns
4. Recommendation

Use concise military briefing style.
"""

    return llm.invoke(prompt).content

In [4]:
from typing import TypedDict

class MissionState(TypedDict):

    report: str

    isr_output: str

    cyber_output: str

    logistics_output: str

    threat_output: str

    planner_output: str

    governance_output: str

    commander_brief: str

In [5]:
def isr_agent(state):

    result = run_agent(
        "ISR Intelligence Officer",
        state["report"]
    )

    return {
        "isr_output": result
    }

In [6]:
def cyber_agent(state):

    result = run_agent(
        "Cyber Intelligence Officer",
        state["report"]
    )

    return {
        "cyber_output": result
    }

In [7]:
def logistics_agent(state):

    logistics_prompt = f"""
Mission Report:

{state['report']}

Assume current operational readiness.

Assess:

- fuel
- supply chain
- mobility
- medical readiness

Identify possible logistics risks.
"""

    result = run_agent(
        "Logistics Officer",
        logistics_prompt
    )

    return {
        "logistics_output": result
    }

In [8]:
def threat_agent(state):

    prompt = f"""
You are Threat Assessment Officer.

ISR Findings:
{state['isr_output']}

Cyber Findings:
{state['cyber_output']}

Logistics Findings:
{state['logistics_output']}

Determine:

GREEN
YELLOW
ORANGE
RED

Explain rationale.

Provide confidence score.
"""

    result = llm.invoke(prompt).content

    return {
        "threat_output": result
    }

In [9]:
def planner_agent(state):

    prompt = f"""
You are Mission Planning Officer.

Threat Assessment:

{state['threat_output']}

Generate 3 Courses Of Action.

For each:

- Objective
- Benefits
- Risks
- Resources Needed

Decision support only.
No autonomous actions.
"""

    result = llm.invoke(prompt).content

    return {
        "planner_output": result
    }

In [10]:
def governance_agent(state):

    prompt = f"""
You are Governance and Compliance Officer.

Review:

{state['planner_output']}

Check:

- policy compliance
- civilian impact
- escalation risk
- ethical concerns

Output:

APPROVED
or

REQUIRES HUMAN REVIEW

Explain reasoning.
"""

    result = llm.invoke(prompt).content

    return {
        "governance_output": result
    }

In [11]:
def debate_agent(state):

    prompt = f"""
Conduct an internal review.

ISR Officer:
{state['isr_output']}

Cyber Officer:
{state['cyber_output']}

Threat Officer:
{state['threat_output']}

Mission Planner:
{state['planner_output']}

Perform a debate:

1. Agreements
2. Disagreements
3. Missing evidence
4. Revised conclusions
"""

    result = llm.invoke(prompt).content

    return {
        "planner_output":
            state["planner_output"]
            + "\n\nDEBATE REVIEW\n\n"
            + result
    }

In [12]:
def commander_agent(state):

    prompt = f"""
Generate Executive Commander Brief.

Mission Report:
{state['report']}

ISR:
{state['isr_output']}

Cyber:
{state['cyber_output']}

Logistics:
{state['logistics_output']}

Threat:
{state['threat_output']}

Mission Plan:
{state['planner_output']}

Governance:
{state['governance_output']}

Format:

Situation Summary

Threat Level

Recommended COAs

Risks

Confidence

Required Human Decisions
"""

    result = llm.invoke(prompt).content

    return {
        "commander_brief": result
    }

In [13]:
from langgraph.graph import StateGraph
from langgraph.graph import END

builder = StateGraph(MissionState)

builder.add_node("isr", isr_agent)
builder.add_node("cyber", cyber_agent)
builder.add_node("logistics", logistics_agent)

builder.add_node("threat", threat_agent)

builder.add_node("planner", planner_agent)

builder.add_node("debate", debate_agent)

builder.add_node("governance", governance_agent)

builder.add_node("commander", commander_agent)

In [14]:
builder.set_entry_point("isr")

builder.add_edge("isr", "cyber")
builder.add_edge("cyber", "logistics")

builder.add_edge("logistics", "threat")

builder.add_edge("threat", "planner")

builder.add_edge("planner", "debate")

builder.add_edge("debate", "governance")

builder.add_edge("governance", "commander")

builder.add_edge("commander", END)

graph = builder.compile()

In [15]:
result = graph.invoke(
    {
        "report": intel_report
    }
)

In [16]:
print("\n=== ISR ===\n")
print(result["isr_output"])

print("\n=== CYBER ===\n")
print(result["cyber_output"])

print("\n=== LOGISTICS ===\n")
print(result["logistics_output"])

print("\n=== THREAT ===\n")
print(result["threat_output"])

print("\n=== PLANNER ===\n")
print(result["planner_output"])

print("\n=== GOVERNANCE ===\n")
print(result["governance_output"])

print("\n=== COMMANDER BRIEF ===\n")
print(result["commander_brief"])


=== ISR ===

**ISR INTELLIGENCE BRIEFING**
**TO:** Command Staff
**FROM:** ISR Intelligence Officer
**DATE:** 2026-06-15
**SUBJECT:** Northern Sector Delta Activity Assessment

---

### 1. FINDINGS

*   **Activity:** Confirmed presence of twelve (12) armored vehicles in Northern Sector Delta.
*   **Deployment:** Temporary encampment established within the last 24 hours, indicating recent arrival or rapid deployment.
*   **Intent/Operational Tempo:** Communications intercepts show a significant spike (+240% increase), suggesting active coordination and operational planning.
*   **Environmental Factors:** Weather conditions are optimal for sustained vehicle movement and operations.
*   **Risk Profile:** Three (3) civilian settlements are located within a 12km radius of the detected activity.

### 2. CONFIDENCE SCORE: 90/100

*(High confidence due to corroboration across multiple ISR platforms—drone feed, satellite imagery, and comms intercepts.)*

### 3. CONCERNS

*   **Escalation Risk:

In [17]:
%%writefile app.py

import streamlit as st

# -----------------------------
# Graph Logic (formerly defense_graph.py)
# -----------------------------
class DefenseGraph:

    def invoke(self, data):
        report = data.get("report", "")

        return {
            "isr_output": self.isr_agent(report),
            "cyber_output": self.cyber_agent(report),
            "logistics_output": self.logistics_agent(report),
            "threat_output": self.threat_agent(report),
            "planner_output": self.planner_agent(report),
            "governance_output": self.governance_agent(report),
            "commander_brief": self.commander_brief(report)
        }

    def isr_agent(self, report):
        return f"ISR Analysis:\n\nIntelligence extracted from report:\n{report}"

    def cyber_agent(self, report):
        return "Cyber Assessment: No critical cyber threats detected."

    def logistics_agent(self, report):
        return "Logistics Assessment: Supply chain and assets available."

    def threat_agent(self, report):
        return "Threat Assessment: Medium risk level identified."

    def planner_agent(self, report):
        return "Mission Plan: Recommend surveillance and monitoring."

    def governance_agent(self, report):
        return "Governance Review: Mission complies with policies."

    def commander_brief(self, report):
        return (
            "Commander Summary:\n"
            "Mission intelligence reviewed. "
            "Threat level moderate. "
            "Recommend proceeding with caution."
        )


# Create graph instance
graph = DefenseGraph()

# -----------------------------
# Streamlit UI
# -----------------------------
st.set_page_config(
    page_title="Joint Operations Center",
    layout="wide"
)

st.title("🛡️ Agentic AI Joint Operations Center")

report = st.text_area(
    "Mission Intelligence",
    height=300,
    placeholder="Enter mission intelligence report..."
)

if st.button("Run Analysis"):

    result = graph.invoke({"report": report})

    col1, col2 = st.columns(2)

    with col1:

        st.subheader("ISR")
        st.write(result["isr_output"])

        st.subheader("Cyber")
        st.write(result["cyber_output"])

        st.subheader("Logistics")
        st.write(result["logistics_output"])

    with col2:

        st.subheader("Threat")
        st.write(result["threat_output"])

        st.subheader("Mission Planning")
        st.write(result["planner_output"])

        st.subheader("Governance")
        st.write(result["governance_output"])

    st.subheader("Commander Brief")
    st.success(result["commander_brief"])

    decision = st.radio(
        "Commander Decision",
        ["Approve", "Reject"]
    )

    st.write("Decision:", decision)

Overwriting app.py


In [18]:
!streamlit run app.py

2026-06-22 18:36:36.107 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.10:8501

^C
  Stopping...
